# Managed Tables Part2

## display schema

In [0]:
%sql
USE workspace.demo;

## display current catalog, schema

In [0]:
%sql
select  current_catalog(), current_schema()

current_catalog(),current_schema()
workspace,demo


## Shallow Clone: same data, no duplicate storage

In [0]:
%sql
CREATE TABLE customers_shallow_clone
SHALLOW CLONE customers;

## Confirm it has the same data

In [0]:
%sql
SELECT * FROM customers_shallow_clone;

id,name,city
1,Ravi,Hyderabad
2,Anita,Bangalore


## Confirm its a real, independent catalog entry

In [0]:
%sql
DESCRIBE TABLE EXTENDED customers_shallow_clone;

col_name,data_type,comment
id,int,null
name,string,null
city,string,null
,,
# Delta Statistics Columns,,
Column Names,"id, name, city",
Column Selection Method,first-32,
,,
# Detailed Table Information,,
Catalog,workspace,


## Pull just the CLONE operation's metrics, isolated

In [0]:
%sql
SELECT operation, operationMetrics
FROM (DESCRIBE HISTORY customers_shallow_clone)
WHERE operation = 'CLONE';

operation,operationMetrics
CLONE,"Map(removedFilesSize -> 0, numRemovedFiles -> 0, sourceTableSize -> 1266, numCopiedFiles -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, copiedFilesSize -> 0, sourceNumOfFiles -> 1)"


## Deep Clone: an actual, independent copy of the data

In [0]:
%sql
CREATE TABLE customers_deep_clone
DEEP CLONE customers;

## Confirm it has the same data

In [0]:
%sql
SELECT * FROM customers_deep_clone;

id,name,city
1,Ravi,Hyderabad
2,Anita,Bangalore


## Pull just the CLONE operation's metrics

In [0]:
%sql
SELECT operation, operationMetrics
FROM (DESCRIBE HISTORY customers_deep_clone)
WHERE operation = 'CLONE';

operation,operationMetrics
CLONE,"Map(removedFilesSize -> 0, numRemovedFiles -> 0, sourceTableSize -> 1266, numCopiedFiles -> 1, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, copiedFilesSize -> 1266, sourceNumOfFiles -> 1)"


## Proof: Is Unity Catalog actually managing this, not the legacy Hive metastore?
## 

In [0]:
%sql
SELECT current_metastore(), current_catalog(), current_schema();

current_metastore(),current_catalog(),current_schema()
aws:us-east-2:d3d4d5f4-3e6a-4e8a-8f8d-4f608e25214b,workspace,demo


### See the governance layer in action

In [0]:
%sql
SHOW GRANTS ON TABLE customers;

Principal,ActionType,ObjectType,ObjectKey
srikanth.enterprise.ai@gmail.com,SELECT,TABLE,workspace.demo.customers


In [0]:
%sql
GRANT SELECT ON TABLE customers TO `srikanth.enterprise.ai@gmail.com`;

## Drop clone tables

In [0]:
%sql
DROP TABLE IF EXISTS customers_shallow_clone;
DROP TABLE IF EXISTS customers_deep_clone;